In [ ]:
# This is run from main working directory
import mlrun
mlrun.set_environment(api_path="http://localhost:30070")
project = mlrun.load_project(name="legalcontractextractor", context="../")
# print(project.to_yaml())

from dotenv import load_dotenv
load_dotenv()

import os
ENV = os.environ["ENV"]

# Run the workflow

In [ ]:
from datetime import datetime
source_path = f"s3://{ENV}-mlops-bucket-haviv/data/raw"
version = datetime.now().strftime("%Y%m%d_%H%M")

output_path = f"s3://{ENV}-mlops-bucket-haviv/data/processed_training"

In [ ]:
# Train dataset
runobj = mlrun.run_function(
    "raw-proc",  # use the function name registered in the register_funcs.ipynb file
    inputs={"input_uri": f"{source_path}/train.parquet"},
    params={
        "label_column": "inference",
        "artifact_key": f"train_data",
        "version": version,
        "output_uri_path": output_path,
    },
    local=True,
)

# Validation dataset
runobj = mlrun.run_function(
    "raw-proc",
    inputs={"input_uri": f"{source_path}/validation.parquet"},
    params={
        "label_column": "inference",
        "artifact_key": f"validation_data",
        "version": version,
        "output_uri_path": output_path,
    },
    local=True,
)

# Test dataset
runobj = mlrun.run_function(
    "raw-proc",
    inputs={"input_uri": f"{source_path}/test.parquet"},
    params={
        "label_column": "inference",
        "artifact_key": f"test_data",
        "version": version,
        "output_uri_path": output_path,
    },
    local=True,
)

In [ ]:
# For kubeflow pipelines: not using during development because of ghost runs and caching issues..for another time
# from datetime import datetime

# run_obj = project.run(
#     name="evaluate_noTrain",
#     arguments={
#         "source_path": "s3://legal-llama-data/raw",
#         "version": datetime.now().strftime("%Y%m%d_%H%M")
#     },
#     local=True,   # Run the pipeline sequence locally
#     watch=True    # Print the progress to the console
# )

## Validation

In [ ]:
from IPython.display import display

# Testing direct access to S3 data (underlying s3fs)
# data_uri = "s3://legal-llama-data/raw/test.parquet"
# df = mlrun.get_dataitem(data_uri).as_df() #this reads into a dataframe and only works if the file is a csv/parquet... jsonl does not work
# display(df.head(1))

data_uri = "store://datasets/legalcontractextractor/raw-proc-process-raw_test_data:latest"
data_pointer = mlrun.get_dataitem(data_uri)
print(data_pointer.url) # S3 path

In [ ]:
# Testing data versioning and access to registered datasets in MLRun
data_uri = "store://datasets/legalcontractextractor/raw-proc-process-raw_test_data:latest"
#data_uri = "store://datasets/finetune-legal-extractor/raw-proc-process-raw_test_data:20260427_1945"

# Fetch the item and immediately convert it to a Pandas DataFrame
df = mlrun.get_dataitem(data_uri).as_df()
print(df.head(1)['inference'][0])


In [ ]:
"""
# I blame mlrun for this weird behaviour, instead of the URI on the UI with the ://files path segment, it uses ://datasets
# Now that we are not using 
artifact_uri = "store://datasets/finetune-legal-extractor/raw-proc-process-raw_test_data:latest"

data_item = mlrun.get_dataitem(artifact_uri)
s3_path = data_item.url
print(s3_path) 

import pandas as pd
import io

raw_bytes = data_item.get()
df = pd.read_json(io.BytesIO(raw_bytes), orient="records", lines=True)

inferences = df.head(1)['inference'][0]
for i in inferences:
    print(i)
"""


In [ ]:
artifact = project.get_artifact(key="raw-proc-process-raw_validation_data", tag="20260816_1514") # use the db-key not key
# this wont work project.get_artifact(key="train_data")

print(artifact.get_store_url())
print(artifact.target_path) # points to the source path of the latest version of the artifact
print(artifact.db_key)
print(artifact.key)

In [ ]:
# all datasets
artifacts = project.list_artifacts()
datasets = [artifact for artifact in artifacts if artifact['kind'] == "dataset"]
for i in datasets:
    print(i['metadata']['key'], i['metadata']['tag'])

# Deleting artifacts directly

In [ ]:
adsasd

In [ ]:
# This deletes all artifacts in the database
db = mlrun.get_run_db()
db.del_artifacts(project=project.metadata.name)

print("all artifacts wiped from " + project.metadata.name)

In [ ]:
project.spec.get_code_path()